# NDgpu — 3D tri-S_N on GPU: level-scheduled prism sweeps (Colab)

Benchmarks the **3D** discrete-ordinates solver (`TriSNTransportSolver` on an
extruded triangular-prism mesh, `engine="levels"`) CPU vs GPU, with mesh
refinement (in-plane `refine` and axial `nz`) as the problem-size axis. This is
the Phase-2 GPU check for the 3D port: the level-scheduled sweep DAG now carries
axial dependency edges (up to 5 inflow faces/cell), replayed as one CUDA graph
per (group, iface); the within-group DSA loop runs device-resident.

The `cudagraph` column confirms capture engaged; CPU and GPU run the identical
iteration sequence, so the speed-up is tolerance-independent. Because 3D sparse
LU blows up, the levels engine is the *only* practical prism sweep at scale --
this measures how its GPU throughput grows with the prism count.

(Cross sections are illustrative placeholders, not predictive.)

In [ ]:
import os
try:                                        # Colab: upload dist/ndgpu-src.zip
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    get_ipython().run_line_magic("pip", f"install -q {zip_name}")
    try:
        import cupy
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q cupy-cuda12x")
    get_ipython().system("nvidia-smi -L")
except ImportError:                         # local run: ndgpu already importable
    pass

In [ ]:
import time
import numpy as np
from ndgpu import Material
from ndgpu.tri import TriGrid
from ndgpu.tri_sn import TriSNTransportSolver

try:
    import cupy
    HAVE_GPU = cupy.cuda.runtime.getDeviceCount() > 0
except Exception:
    HAVE_GPU = False
print("GPU available:", HAVE_GPU)

QUICK = bool(os.environ.get("NDGPU_QUICK"))
# (in-plane rows/cols, nz). A T4 (16 GB) handles the larger ones; keep nz modest.
SIZES = [(8, 4), (12, 6)] if QUICK else [(8, 4), (12, 6), (16, 8), (20, 10)]
TOL = dict(tol_k=5e-7, tol_source=5e-6, max_outer=200)
QUAD = dict(n_polar=4, n_azi=8)
MAT = Material(diffusion=[1.1], sigma_a=[0.015], nu_sigma_f=[0.022],
               sigma_s=[[0.0]], name="fuel")   # mildly multiplying prism block


def run(nrc, nz, device):
    grid = TriGrid(shape=(nrc, nrc, 2, nz), side=3.0, height=3.0 * nz)  # ~cubic cells
    t = time.perf_counter()
    s = TriSNTransportSolver(grid, MAT, bc="vacuum", engine="levels",
                             device=device, **QUAD)
    setup = time.perf_counter() - t
    r = s.solve(**TOL)
    assert r.converged, f"nrc={nrc} nz={nz} {device} not converged"
    return dict(cells=int(np.prod(grid.shape)), k=r.k_eff, outers=r.outer_iterations,
                sweeps=r.n_sweeps, setup=setup, solve=r.solve_seconds,
                graphs=s.graphs_active, gerr=s._graph_error)


if HAVE_GPU:                                # warm-up (compile kernels, capture)
    run(8, 4, "gpu")

In [ ]:
rows = []
for nrc, nz in SIZES:
    cpu = run(nrc, nz, "cpu")
    row = dict(nrc=nrc, nz=nz, **cpu)
    if HAVE_GPU:
        gpu = run(nrc, nz, "gpu")
        row.update(k_gpu=gpu["k"], t_gpu=gpu["solve"], graphs=gpu["graphs"],
                   gerr=gpu["gerr"], speedup=cpu["solve"] / gpu["solve"])
    rows.append(row)

hdr = f"{'nrc':>4} {'nz':>3} {'cells':>7} {'outers':>6} {'sweeps':>6} {'k':>9} {'t_cpu[s]':>9}"
if HAVE_GPU:
    hdr += f" {'t_gpu[s]':>9} {'dpcm':>6} {'speedup':>8} {'cudagraph':>9}"
print(hdr)
for r in rows:
    line = (f"{r['nrc']:>4d} {r['nz']:>3d} {r['cells']:>7d} {r['outers']:>6d} "
            f"{r['sweeps']:>6d} {r['k']:>9.5f} {r['solve']:>9.2f}")
    if HAVE_GPU:
        cg = {True: 'on', False: 'fallback'}.get(r.get('graphs'), '?')
        line += (f" {r['t_gpu']:>9.2f} {(r['k_gpu']-r['k'])*1e5:>6.2f} "
                 f"{r['speedup']:>8.2f} {cg:>9}")
    print(line)
if HAVE_GPU and any(r.get('graphs') is False for r in rows):
    print("cudagraph fallback reason:", next(r['gerr'] for r in rows if r.get('graphs') is False))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
cx = [r['cells'] for r in rows]
ax[0].plot(cx, [r['solve'] for r in rows], 'o-', label='CPU')
if HAVE_GPU:
    ax[0].plot(cx, [r['t_gpu'] for r in rows], 's-', label='GPU')
ax[0].set(xlabel='prisms (cells)', ylabel='solve wall [s]', xscale='log',
          yscale='log', title='3D prism sweep: cost vs size'); ax[0].legend(); ax[0].grid(True, which='both', alpha=0.3)
if HAVE_GPU:
    ax[1].axhline(1.0, color='k', lw=0.8, ls='--')
    ax[1].plot(cx, [r['speedup'] for r in rows], 'D-', color='C2')
    ax[1].set(xlabel='prisms (cells)', ylabel='GPU speed-up (x)', xscale='log',
              title='CPU/GPU crossover'); ax[1].grid(True, which='both', alpha=0.3)
plt.show()

## Accuracy — step vs second-order SCB (CPU)

The throughput above uses step differencing (the levels/GPU engine). For
*accuracy*, second-order **SCB** (`scheme="scb"`) resolves the flux far better
at equal prism resolution -- on HP-MR this is what gets the drum-worth sign
right at coarse mesh. SCB runs on the LU engine (CPU) for now; the level-
scheduled SCB prism sweep (for GPU) is the remaining piece.

In [ ]:
from ndgpu.tri import TriGrid as _TG
def _k(scheme, nrc, nz):
    g = _TG(shape=(nrc, nrc, 2, nz), side=12.0 / nrc, height=12.0)
    return TriSNTransportSolver(g, MAT, n_polar=2, n_azi=8, bc="vacuum",
                                scheme=scheme, engine="lu").solve(
        tol_k=1e-7, tol_source=1e-6, max_outer=800).k_eff
ref = _k("scb", 12, 12)                      # fine-mesh reference
es, ec = abs(_k("step", 6, 6) - ref), abs(_k("scb", 6, 6) - ref)
print(f"fine-mesh ref k = {ref:.5f}")
print(f"coarse (6x6x6) step error = {es*1e5:6.0f} pcm")
print(f"coarse (6x6x6) SCB  error = {ec*1e5:6.0f} pcm   ({es/ec:.2f}x more accurate)")

## Reading the results

* **`cudagraph = on`** confirms the prism level loop (with axial edges) is
  replayed as one captured launch per group; `fallback` prints the reason.
* **`dpcm`** (GPU k - CPU k) should be a few pcm at most -- identical iteration
  sequence, only device floating-point associativity differs.
* **speed-up vs cells**: the level count grows ~ mesh diameter while the work
  per level grows with the prism count, so the GPU advantage should widen with
  refinement -- and unlike diffusion there is no O(N^1.5) LU wall to fall back
  on, so the levels engine is the practical prism sweep at scale.

Next (Phase 4): 3D CMFD, then the 3D host-LU-vs-multigrid CMFD-solve bake-off --
the regime where 3D LU's O(N^2) cost is exactly why the multigrid port matters.